# 7주차 — DDL — 테이블 설계와 생성
**의료진을 위한 데이터베이스** · 미니 EMR 실습 노트북

사용법: 위에서부터 순서대로 셀을 실행하세요 (`Shift+Enter`).
`q("SELECT ...")` 안의 SQL만 바꿔가며 실험하면 됩니다. 원본은 언제든 다시 내려받을 수 있으니 마음껏 실험하세요.

In [ ]:
# [준비] 미니 EMR 데이터베이스 내려받기 + 조회 도우미 정의 — 이 셀을 가장 먼저 실행!
import os, urllib.request, sqlite3
import pandas as pd

DB_URL = "https://raw.githubusercontent.com/kevinkwshin/medical-db-course-2026/main/mini_emr/mini_emr.sqlite"   # 강의자 공지 주소
if not os.path.exists("mini_emr.sqlite"):
    try:
        urllib.request.urlretrieve(DB_URL, "mini_emr.sqlite")
        print("mini_emr.sqlite 다운로드 완료")
    except Exception as e:
        print("자동 다운로드 실패 — 왼쪽 파일 탭에서 mini_emr.sqlite를 직접 업로드하세요.", e)

con = sqlite3.connect("mini_emr.sqlite")

def q(sql):
    """SQL을 실행하고 결과를 표로 반환"""
    return pd.read_sql_query(sql, con)

def run(sql):
    """결과를 반환하지 않는 SQL(INSERT/UPDATE/CREATE 등) 실행"""
    con.executescript(sql); con.commit(); print("실행 완료")

q("SELECT name AS 테이블 FROM sqlite_master WHERE type='table' ORDER BY 1")

## 1. CREATE TABLE — 제약조건과 함께
(이 노트북에서는 실습용 사본 위에서 자유롭게 만들었다 지워도 됩니다)

In [ ]:
# 코호트 테이블 생성
run("""
CREATE TABLE IF NOT EXISTS my_cohort (
  subject_id   TEXT PRIMARY KEY,
  patient_id   TEXT NOT NULL REFERENCES patient(patient_id),
  index_date   DATE NOT NULL,
  age_at_index INTEGER CHECK (age_at_index BETWEEN 0 AND 120),
  sex          TEXT NOT NULL CHECK (sex IN ('M','F')),
  dm_flag      INTEGER NOT NULL DEFAULT 0 CHECK (dm_flag IN (0,1))
);
""")

In [ ]:
# 생성 확인
q("""
SELECT name FROM sqlite_master WHERE type='table' ORDER BY 1
""")

## 2. 제약조건이 일하는 순간 — 아래 셀은 **일부러 에러**가 납니다

In [ ]:
# CHECK 위반 실험: SpO2 250%는 입력 자체가 거부된다
try:
    run("""
    CREATE TABLE IF NOT EXISTS vital_demo (visit_id TEXT, spo2 INTEGER CHECK (spo2 BETWEEN 0 AND 100));
    INSERT INTO vital_demo VALUES ('V0001', 250);
    """)
except Exception as e:
    print('거부됨 →', e)

## 3. ALTER / DROP

In [ ]:
# 컬럼 추가 후 테이블 정리
run("""
ALTER TABLE my_cohort ADD COLUMN htn_flag INTEGER DEFAULT 0;
DROP TABLE IF EXISTS vital_demo;
""")

## 연습문제

In [ ]:
# [연습 Q1] '수술(operation)' 테이블을 설계해 CREATE 하라 — 수술ID(PK), 내원ID(FK), 수술명(NOT NULL), 수술일, 집도의ID(FK)
# 힌트: run() 사용
q("""
-- 여기에 SQL을 작성하세요

""")

In [ ]:
# [연습 Q2] operation에 마취종류 컬럼을 ALTER로 추가하고, CHECK로 ('전신','부위','국소')만 허용해 보라 — SQLite에서 기존 테이블에 CHECK 추가가 왜 어려운지 겪어보기
q("""
-- 여기에 SQL을 작성하세요

""")

In [ ]:
# [연습 Q3] 자기 연구 주제의 코호트 테이블을 컬럼 6개 이상으로 설계하라 (과제 제출용)
q("""
-- 여기에 SQL을 작성하세요

""")

---
## 정답 및 해설
먼저 스스로 푼 다음에 열어 보세요.

**A1. 수술 테이블**
```sql
CREATE TABLE operation (
  op_id TEXT PRIMARY KEY,
  visit_id TEXT NOT NULL REFERENCES visit(visit_id),
  op_name TEXT NOT NULL,
  op_date DATE,
  surgeon_id TEXT REFERENCES staff(staff_id)
);
```

**A2. 마취종류 추가**
```sql
-- SQLite의 ALTER는 CHECK 추가를 지원하지 않음 → 테이블 재생성 필요.
-- 교훈: 제약조건은 처음 설계할 때 넣는 것이 훨씬 싸다.
ALTER TABLE operation ADD COLUMN anesthesia TEXT;
```